# 80 — Blind-A responder swap: Gemini-generated `predicted_response` -> submission zip

Takes the existing Blind-A prediction (whose `predicted_track_ids` came from the
validated retrieval+rerank pipeline, e.g. config 194 via colab/41) and regenerates
ONLY `predicted_response` with the Gemini API, then packages the CodaBench zip.
`predicted_track_ids` are left untouched -> nDCG/diversity axes unchanged; this only
moves the LLM axis (0.30 of the composite).

No GPU needed (no local responder, no retrieval). Prereqs: `GEMINI_API_KEY` in Colab
secrets, and an existing Blind-A `predicted_track_ids` prediction.json (from colab/41
or on Drive). top_n=1 by default (v5-kto's winning setting; not assumed optimal for
Gemini -- A/B 1 vs 3 on dev later via nb79's judge).

Rules note: external LLM APIs aren't banned, but final code must be uploaded
(due 2026-07-09) and the judge family is Gemini (self-preference risk). See
project_responder_topn_ab_status_2026_06_04 memory.


In [ ]:
# 1) Setup — clone branch + Gemini key + Drive + light deps (no GPU).
import os
os.environ['USE_FLAX'] = '0'; os.environ['USE_TF'] = '0'
from google.colab import userdata, drive
os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY')          # add this Colab secret first
os.environ.setdefault('GEMINI_RESPONDER_MODEL', 'gemini-2.5-pro')  # known-good generator (~4.05); flash-lite regressed the LLM axis
os.environ.setdefault('GEMINI_JUDGE_MODEL', 'gemini-2.5-flash')           # JUDGE/select best-of-N with flash
drive.mount('/content/drive', force_remount=False)
BRANCH = 'recall-union-lgbm'
!rm -rf /content/recsys2026
!git clone -b {BRANCH} https://github.com/orrimoch/recsys2026-lora-tutorial.git /content/recsys2026
%cd /content/recsys2026
!pip install -q -U google-generativeai 'datasets' 'pandas<3.0'
print('setup done | GEMINI key present:', bool(os.environ.get('GEMINI_API_KEY')))

In [ ]:
# 2) Locate the existing Blind-A prediction (predicted_track_ids to KEEP).
#    Accepts a raw prediction.json OR a CodaBench submission zip.
import os, json, zipfile, glob

# === EXP-001 parameters (config-203 track_ids + Gemini responder swap) ===
EXP_ID    = "EXP-001"
CONFIG_ID = 203
TOP_N     = 1   # known-good (v5-kto winning setting); config 203 itself sets top_n_for_prompt=1.
                # (TOP_N=3 was speculative — Gemini may dilute across 3 tracks — and slower.)
BLIND_DATASET = 'talkpl-ai/TalkPlayData-Challenge-Blind-A'

# SRC = YOUR config-203 Blind-A prediction.json (or zip) carrying predicted_track_ids to KEEP.
# If you don't have one yet, run colab/41_run_blindset_A.ipynb with config 203 first, then point here.
SRC = '/content/drive/MyDrive/recsys2026/203-union-sasrec-pg-colbert-claptext-v5kto-blindA.json'  # <-- set to your 203 prediction path

PRED_IN  = '/content/recsys2026/music-crs-baselines/exp/inference/blindset_A/blindA_candidates.json'
PRED_OUT = '/content/recsys2026/music-crs-baselines/exp/inference/blindset_A/blindA_gemini.json'
os.makedirs(os.path.dirname(PRED_IN), exist_ok=True)

if not os.path.exists(SRC):
    print('SRC not found. Candidates seen under MyDrive:',
          glob.glob('/content/drive/MyDrive/**/*blindA*.*', recursive=True))
    raise FileNotFoundError(f'Set SRC to your config-203 candidates file. Looked for: {SRC}')

if SRC.endswith('.zip'):
    with zipfile.ZipFile(SRC) as zf:
        name = 'prediction.json' if 'prediction.json' in zf.namelist() else zf.namelist()[0]
        rows = json.loads(zf.read(name))
else:
    rows = json.load(open(SRC))

assert isinstance(rows, list) and all('predicted_track_ids' in r for r in rows), 'need predicted_track_ids'
json.dump(rows, open(PRED_IN, 'w'), ensure_ascii=False)
print(f'using {SRC}\n -> {len(rows)} rows (expect 80), wrote {PRED_IN}')

In [ ]:
# 3) Regenerate predicted_response over Blind-A via Gemini (track_ids untouched).
#    On any per-row API failure the script keeps that row's original response.
# REVERTED to the known-good 0.47/4.05 responder: plain prompt + pro generation + best-of-3.
# (The anchored-rubric / prompt-enrichment / lite-model complications dropped LLM to ~3.2.)
STRUCTURED_PERSONALITY = False  # OFF: clean test regressed it (0.43->0.40). plain wins.
BEST_OF = 3        # proven (best-of-3 -> 0.47/0.49); set 1 for single-shot (~4.05)
SMOKE   = False    # EXP-001: full 80-row run (smoke pass skipped per operator). best-of-3 over 80 rows.
# COST: best-of-3 = 3 pro GEN + 3 flash JUDGE calls per row. pro is the known-good generator;
# switch GEMINI_RESPONDER_MODEL (cell 1) to a flash model to cut cost (at some LLM-axis risk).
limit = '--limit 5' if SMOKE else ''
flags = ('--structured-personality ' if STRUCTURED_PERSONALITY else '') + limit
!cd /content/recsys2026 && python -u scripts/gemini_responder.py \
    --pred {PRED_IN} --out {PRED_OUT} \
    --dataset {BLIND_DATASET} --top-n {TOP_N} --best-of {BEST_OF} --judge-model gemini-2.5-flash --sleep 0.2 {flags}
if SMOKE:
    print('\n*** SMOKE run (5 rows). Set SMOKE=False and re-run before packaging (cell 4). ***')
import json
out = json.load(open(PRED_OUT))
print(f'\nrows out: {len(out)}')
print('sample response:\n', out[0]['predicted_response'][:500])

In [ ]:
# 4) Validate schema (blindA) + package the CodaBench zip (root = prediction.json).
from datetime import date
import os, sys, zipfile, shutil
sys.path.insert(0, '/content/recsys2026/scripts')
from validate_prediction import load_prediction, validate_schema, package_zip

predictions = load_prediction(PRED_OUT)
errors = validate_schema(predictions, 'blindA')
if errors:
    print('Schema validation FAILED:')
    for e in errors[:20]: print('  -', e)
    raise SystemExit('Refusing to package — fix and rerun.')
print(f'schema OK ({len(predictions)} rows for blindA — expected 80)')

ZIP_PATH = f'/content/recsys2026/data/submissions/blindset_A_{date.today().isoformat()}_{CONFIG_ID}_gemini.zip'
os.makedirs(os.path.dirname(ZIP_PATH), exist_ok=True)
out_zip = package_zip(PRED_OUT, ZIP_PATH)
with zipfile.ZipFile(out_zip) as zf:
    members = zf.namelist()
assert members == ['prediction.json'], f'wrong zip layout: {members}'
print('packaged ->', out_zip, '| contains', members)

drive_zip = f'/content/drive/MyDrive/blindset_runs/{os.path.basename(ZIP_PATH)}'
os.makedirs(os.path.dirname(drive_zip), exist_ok=True)
shutil.copy(out_zip, drive_zip)
print('Drive copy ->', drive_zip)
print('\nUpload this zip to CodaBench.')

In [ ]:
# RESULTS_JSON emitter — run after cell 4 (zip packaged).
# nDCG@20, cat_div, lex_div, and llm_judge come from CodaBench after uploading the zip.
# Fill in the four variables below with the CodaBench result, then re-run this cell.
# The emitter prints a machine-parseable RESULTS_JSON block to paste back.
import sys, os
for _p in ("/content/recsys2026", os.getcwd(), os.path.dirname(os.getcwd())):
    if os.path.isdir(os.path.join(_p, "scripts")):
        sys.path.insert(0, _p); break
from scripts.emit_results import print_results_block

exp = globals().get("EXP_ID", "EXP-UNSET")
config = globals().get("CONFIG_ID", globals().get("CONFIG", 194))

# --- Fill these in from the CodaBench result page after uploading the zip ---
BLIND_NDCG      = None   # float e.g. 0.44  (nDCG@20 from the leaderboard)
BLIND_CAT_DIV   = None   # float e.g. 0.03  (Catalog Diversity)
BLIND_LEX_DIV   = None   # float e.g. 0.79  (Lexical Diversity)
BLIND_LLM_JUDGE = None   # float e.g. 4.20  (LLM Judge mean 1-5; None until score lands)
# ---------------------------------------------------------------------------

# n_sessions: `out` is the list loaded in cell 3 (80 Blind-A rows).
_n = len(out) if 'out' in dir() else 80   # fallback to 80 (expected Blind-A size)

if BLIND_NDCG is None:
    print("[emitter] BLIND_NDCG not set — fill in from CodaBench then re-run.")
else:
    print_results_block(
        exp=exp,
        config=int(config),
        ndcg=BLIND_NDCG,
        cat_div=BLIND_CAT_DIV if BLIND_CAT_DIV is not None else 0.0,
        lex_div=BLIND_LEX_DIV if BLIND_LEX_DIV is not None else 0.0,
        llm_judge=BLIND_LLM_JUDGE,
        n_sessions=_n,
        gate="blindA",
    )